# **dlt sources and resources**: Create your first dlt pipeline  [![Open in molab](https://marimo.io/molab-shield.svg)](https://molab.marimo.io/github/dlt-hub/dlt/blob/master/docs/education/dlt-fundamentals-course/lesson_2_dlt_sources_and_resources_create_first_dlt_pipeline.py) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dlt-hub/dlt/blob/master/docs/education/dlt-fundamentals-course/lesson_2_dlt_sources_and_resources_create_first_dlt_pipeline.ipynb) [![GitHub badge](https://img.shields.io/badge/github-view_source-2b3137?logo=github)](https://github.com/dlt-hub/dlt/blob/master/docs/education/dlt-fundamentals-course/lesson_2_dlt_sources_and_resources_create_first_dlt_pipeline.ipynb)


![Lesson_2_dlt_sources_and_resources_Create_first_dlt_pipeline_img1.png](https://storage.googleapis.com/dlt-blog-images/dlt-fundamentals-course/Lesson_2_dlt_sources_and_resources_Create_first_dlt_pipeline_img1.png)

## Recap of [Lesson 1](https://github.com/dlt-hub/dlt/blob/master/docs/education/dlt-fundamentals-course/lesson_1_quick_start.ipynb) 👩‍💻🚀
1. Created a pipeline, loaded toy data into DuckDB, and viewed the load info.
2. Used the `dlt.pipeline` and `pipeline.run` methods.
3. Queried data and viewed tables with DuckDB, the `sql_client`, and the dlt `dataset`.

Now, let's move on to the next lesson to learn more! 🚀

**Here, you will learn how to:**
- Run a simple pipeline with different types of data, such as dataframes, databases and REST APIs.
- Use `dlt.resource`, `dlt.source` and `dlt.transformer`.
- Build your first dlt pipeline for a REST API.

---
## **Install dlt**

In [1]:
%%capture
!pip install "dlt[duckdb,parquet]"

---
## **`dlt` resources**

---
### List of dicts


In the previous lesson, we simply used a list of dictionaries that essentially represents the `pokemon` table.



In [12]:
import dlt

# Sample data containing pokemon details
data = [
    {"id": "1", "name": "bulbasaur", "size": {"weight": 6.9, "height": 0.7}},
    {"id": "4", "name": "charmander", "size": {"weight": 8.5, "height": 0.6}},
    {"id": "25", "name": "pikachu", "size": {"weight": 6, "height": 0.4}},
]


# Set pipeline name, destination, and dataset name
pipeline = dlt.pipeline(
    pipeline_name="quick_start",
    destination="duckdb",
    dataset_name="mydata",
)

# Run the pipeline with data and table name
load_info = pipeline.run(data, table_name="pokemon")

print(load_info)

Pipeline quick_start load step finished in 0.11 seconds
1 load package(s) were loaded to destination duckdb and into dataset mydata
The duckdb destination used duckdb:////content/quick_start.duckdb location to store data
Load package 1790856234.3498142 is LOADED and contains no failed jobs


A better way is to wrap it in the `@dlt.resource` decorator which denotes a logical grouping of data within a data source, typically holding data of similar structure and origin:

In [13]:
import dlt
from dlt.common.typing import TDataItems, TDataItem


# Create a dlt resource from the data
@dlt.resource(table_name="pokemon_new")
def my_dict_list() -> TDataItems:
    yield data

Commonly used arguments:

* **`name`**: The resource name and the name of the table generated by this resource. Defaults to the decorated function name.
* **`table_name`**: The name of the table, if different from the resource name.
* **`write_disposition`**: Controls how to write data to a table. Defaults to the value "append".

> **Why is it a better way?** This allows you to use `dlt` functionalities to the fullest that follow Data Engineering best practices, including incremental loading and data contracts.

Try running the pipeline with the `my_dict_list` resource:

In [14]:
# Run the pipeline and print load info
load_info = pipeline.run(my_dict_list)
print(load_info)

Pipeline quick_start load step finished in 0.12 seconds
1 load package(s) were loaded to destination duckdb and into dataset mydata
The duckdb destination used duckdb:////content/quick_start.duckdb location to store data
Load package 1790856234.5925274 is LOADED and contains no failed jobs


Check what was loaded to the `pokemon_new` table:

In [15]:
pipeline.dataset().pokemon_new.df()

,id,name,size__weight,size__height,_dlt_load_id,_dlt_id
0,1,bulbasaur,6.9,0.7,1790856206.0813408,9mzt2sFklPcozA
1,4,charmander,8.5,0.6,1790856206.0813408,fRGJqWQEaRFnuQ
2,25,pikachu,6.0,0.4,1790856206.0813408,rtpcaVGjGasByw
3,1,bulbasaur,6.9,0.7,1790856234.5925274,FBFd/jWZMFHlJw
4,4,charmander,8.5,0.6,1790856234.5925274,NK0l7A1AcHa/qw
5,25,pikachu,6.0,0.4,1790856234.5925274,NDGrX/mE7qAK8Q


Instead of a dict list, the data could also be a/an:
- dataframe
- database query response
- API request response
- Anything you can transform into JSON/dict format

---
### Dataframes
To create a pipeline using dataframes, you would do:

In [16]:
import pandas as pd


# Define a resource to load data from a CSV
@dlt.resource(table_name="df_data")
def my_df() -> TDataItems:
    sample_df = pd.read_csv("https://people.sc.fsu.edu/~jburkardt/data/csv/hw_200.csv")
    yield sample_df


# Run the pipeline with the defined resource
load_info = pipeline.run(my_df)
print(load_info)

# Query the loaded data from 'df_data'
pipeline.dataset().df_data.df()

Pipeline quick_start load step finished in 0.12 seconds
1 load package(s) were loaded to destination duckdb and into dataset mydata
The duckdb destination used duckdb:////content/quick_start.duckdb location to store data
Load package 1790856234.895318 is LOADED and contains no failed jobs


,index,height_inchesx,_weight_poundsx
0,1,65.78,112.99
1,2,71.52,136.49
2,3,69.40,153.03
3,4,68.22,142.34
4,5,67.79,144.30
...,...,...,...
595,196,65.80,120.84
596,197,66.11,115.78
597,198,68.24,128.30
598,199,68.02,127.47


---
### Databases

To create a pipeline from an SQL database query you would:

In [17]:
%%capture
!pip install pymysql

1. Create and run a pipeline to fetch data from an SQL resource and query the loaded data as follows:

In [18]:
import dlt
from sqlalchemy import create_engine


# Define a resource to fetch genome data from the database
@dlt.resource(table_name="genome_data")
def get_genome_data() -> TDataItems:
    engine = create_engine(
        "mysql+pymysql://rfamro@mysql-rfam-public.ebi.ac.uk:4497/Rfam"
    )
    with engine.connect() as conn:
        query = "SELECT * FROM genome LIMIT 1000"
        rows = conn.execution_options(yield_per=100).exec_driver_sql(query)
        yield from map(lambda row: dict(row._mapping), rows)


# Run the pipeline with the genome data resource
load_info = pipeline.run(get_genome_data)
print(load_info)

# Query the loaded data from 'genome_data'
pipeline.dataset().genome_data.df()

Pipeline quick_start load step finished in 0.93 seconds
1 load package(s) were loaded to destination duckdb and into dataset mydata
The duckdb destination used duckdb:////content/quick_start.duckdb location to store data
Load package 1790856239.690442 is LOADED and contains no failed jobs


,upid,description,total_length,ncbi_id,scientific_name,kingdom,num_rfam_regions,num_families,is_reference,is_representative,...,assembly_acc,assembly_version,assembly_level,ungapped_length,assembly_name,study_ref,circular,wgs_acc,wgs_version,common_name
0,RG000000001,Potato spindle tuber viroid,4591,12892,Potato spindle tuber viroid,viroids,0,0,1,0,...,None,<NA>,None,<NA>,None,None,<NA>,None,<NA>,None
1,RG000000002,Columnea latent viroid,370,12901,Columnea latent viroid,viroids,0,0,1,0,...,None,<NA>,None,<NA>,None,None,<NA>,None,<NA>,None
2,RG000000003,Tomato apical stunt viroid-S,360,53194,Tomato apical stunt viroid-S,viroids,0,0,1,0,...,None,<NA>,None,<NA>,None,None,<NA>,None,<NA>,None
3,RG000000004,Tomato apical stunt viroid,360,12885,Tomato apical stunt viroid,viroids,0,0,1,0,...,None,<NA>,None,<NA>,None,None,<NA>,None,<NA>,None
4,RG000000005,Cucumber yellows virus,7899,32618,Cucumber yellows virus,viruses,0,0,1,0,...,None,<NA>,None,<NA>,None,None,<NA>,None,<NA>,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2995,UP000001264,None,40275,673838,Enterococcus phage phiFL3B,viruses,0,0,0,1,...,GCA_002630665.1,1,complete-genome,<NA>,ASM263066v1,None,<NA>,None,<NA>,None
2996,UP000001265,Enterococcus phage phiFL4A,37856,673839,Enterococcus phage phiFL4A,viruses,2,2,0,1,...,,0,,0,,,0,,0,
2997,UP000001266,Staphylococcus phage SA1,147303,694060,Staphylococcus phage SA1,viruses,0,0,0,1,...,,0,,0,,,0,,0,
2998,UP000001270,Mycobacterium phage AnnaL29,53253,1076630,Mycobacterium phage AnnaL29,viruses,0,0,0,1,...,,0,,0,,,0,,0,


---
### REST API

For REST API endpoints, create a pipeline as follows:

In [19]:
from dlt.sources.helpers import requests


# Define a resource to fetch pokemons from PokeAPI
@dlt.resource(table_name="pokemon_api")
def get_pokemon() -> TDataItems:
    url = "https://pokeapi.co/api/v2/pokemon"
    response = requests.get(url)
    yield response.json()["results"]


# Run the pipeline using the defined resource
load_info = pipeline.run(get_pokemon)
print(load_info)

# Query the loaded data from 'pokemon_api' table
pipeline.dataset().pokemon_api.df()

Pipeline quick_start load step finished in 0.15 seconds
1 load package(s) were loaded to destination duckdb and into dataset mydata
The duckdb destination used duckdb:////content/quick_start.duckdb location to store data
Load package 1790856244.0040455 is LOADED and contains no failed jobs


,name,url,_dlt_load_id,_dlt_id
0,bulbasaur,https://pokeapi.co/api/v2/pokemon/1/,1790856223.0161142,wMHkOXBF7VCXtA
1,ivysaur,https://pokeapi.co/api/v2/pokemon/2/,1790856223.0161142,47YL0R46il1MzA
2,venusaur,https://pokeapi.co/api/v2/pokemon/3/,1790856223.0161142,7C03zEd8ACkc3w
3,charmander,https://pokeapi.co/api/v2/pokemon/4/,1790856223.0161142,Shfqcqzxi8NKLg
4,charmeleon,https://pokeapi.co/api/v2/pokemon/5/,1790856223.0161142,2FB2MpxChL7XZQ
5,charizard,https://pokeapi.co/api/v2/pokemon/6/,1790856223.0161142,zSjk62d0KF6pnw
6,squirtle,https://pokeapi.co/api/v2/pokemon/7/,1790856223.0161142,aAa4LyjdpD0Uqw
7,wartortle,https://pokeapi.co/api/v2/pokemon/8/,1790856223.0161142,X5ZfP2yqZAryTA
8,blastoise,https://pokeapi.co/api/v2/pokemon/9/,1790856223.0161142,PlAUyHU5uTVbsQ
9,caterpie,https://pokeapi.co/api/v2/pokemon/10/,1790856223.0161142,KgKekIE/TpJOEQ


Try loading everything above, in a single pipeline:

In [20]:
load_info = pipeline.run([my_df, get_genome_data, get_pokemon])
print(load_info)

Pipeline quick_start load step finished in 0.84 seconds
1 load package(s) were loaded to destination duckdb and into dataset mydata
The duckdb destination used duckdb:////content/quick_start.duckdb location to store data
Load package 1790856244.460197 is LOADED and contains no failed jobs


Check which new tables were created:

In [21]:
# List all table names from the database
with pipeline.sql_client() as client:
    with client.execute_query(
        "SELECT table_name FROM information_schema.tables"
    ) as table:
        print(table.df())

            table_name
0              df_data
1          genome_data
2              pokemon
3          pokemon_api
4          pokemon_new
5           _dlt_loads
6  _dlt_pipeline_state
7         _dlt_version



---
## **`dlt` sources**

Now that there are multiple `dlt` resources, each corresponding to a separate table, we can group them into a `dlt` source.


![Lesson_2_dlt_sources_and_resources_Create_first_dlt_pipeline_img2](https://storage.googleapis.com/dlt-blog-images/dlt-fundamentals-course/Lesson_2_dlt_sources_and_resources_Create_first_dlt_pipeline_img2.png)

A source is a logical grouping of resources, e.g., endpoints of a single API. The most common approach is to define it in a separate Python module.

* A source is a function decorated with `@dlt.source` that returns one or more resources.
* A source can optionally define a schema with tables, columns, performance hints, and more.
* The source Python module typically contains optional customizations and data transformations.
* The source Python module typically contains the authentication and pagination code for a particular API.

Read more about [sources](https://dlthub.com/docs/general-usage/source) and [resources](https://dlthub.com/docs/general-usage/resource).

You declare a source by decorating a function that returns or yields one or more resources with `@dlt.source`.

Here's how it's done:

In [22]:
from typing import Iterable
from dlt.extract import DltResource


@dlt.source
def all_data() -> Iterable[DltResource]:
    return my_df, get_genome_data, get_pokemon

Only using the source above, load everything into a separate database using a new pipeline:

In [23]:
# Create a pipeline
new_pipeline = dlt.pipeline(
    pipeline_name="resource_source_new", destination="duckdb", dataset_name="all_data"
)

# Run the pipeline
load_info = new_pipeline.run(all_data())

# Print load info
print(load_info)

Pipeline resource_source_new load step finished in 0.36 seconds
1 load package(s) were loaded to destination duckdb and into dataset all_data
The duckdb destination used duckdb:////content/resource_source_new.duckdb location to store data
Load package 1790856384.7798698 is LOADED and contains no failed jobs


> **Why does this matter?**:
- It is more efficient than running your resources separately.
- It organizes both your schema and your code. 🙂
- It enables the option for parallelization.

---
## **`dlt` transformers**

We now know that `dlt` resources can be grouped into a `dlt` source, represented as:


```
                  Source
               /          \
          Resource 1  ...  Resource N

```

However, imagine a scenario where you need an additional step in between:

```
                  Source
                 /     \
             step        \
             /             \
        Resource 1  ...  Resource N

```

This step could arise, for example, in a situation where:

- Resource 1 returns a list of pokemons IDs, and you need to use each of those IDs to retrieve detailed information about the pokemons from a separate API endpoint.

In such cases, you would use `dlt` transformers — special `dlt` resources that can be fed data from another resource:

```
                  Source
                 /     \
          Transformer    \
             /             \
        Resource 1  ...  Resource N

```


Let’s assume Resource 1 is:

In [24]:
@dlt.resource(table_name="pokemon")
def my_pokemons() -> TDataItems:
    pokemons = [
        {"id": "1", "name": "bulbasaur", "size": {"weight": 6.9, "height": 0.7}},
        {"id": "4", "name": "charmander", "size": {"weight": 8.5, "height": 0.6}},
        {"id": "25", "name": "pikachu", "size": {"weight": 6, "height": 0.4}},
    ]
    yield pokemons

We need to get detailed information about pokemons from [PokeAPI](https://pokeapi.co/) `"https://pokeapi.co/api/v2/pokemon/{id}"` based on their IDs. We would do:

In [25]:
# Define a transformer to enrich pokemon data with additional details
# NOTE: the `items` argument contains data from the `my_dict_list` resource
@dlt.transformer(data_from=my_pokemons, table_name="detailed_info")
def poke_details(
    items: TDataItems,
) -> TDataItems:
    for item in items:
        print(f"Item: {item}\n")

        item_id = item["id"]
        url = f"https://pokeapi.co/api/v2/pokemon/{item_id}"
        response = requests.get(url)
        details = response.json()

        print(f"Details: {details}\n")

        yield details


# Set pipeline name, destination, and dataset name
another_pipeline = dlt.pipeline(
    pipeline_name="quick_start",
    destination="duckdb",
    dataset_name="pokedata",
    dev_mode=True,
)

Run the pipeline:

In [26]:
load_info = another_pipeline.run(poke_details())
print(load_info)

Item: {'id': '1', 'name': 'bulbasaur', 'size': {'weight': 6.9, 'height': 0.7}}

Details: {'id': 1, 'name': 'bulbasaur', 'base_experience': 64, 'height': 7, 'is_default': True, 'order': 1, 'weight': 69, 'abilities': [{'is_hidden': False, 'slot': 1, 'ability': {'name': 'overgrow', 'url': 'https://pokeapi.co/api/v2/ability/65/'}}, {'is_hidden': True, 'slot': 3, 'ability': {'name': 'chlorophyll', 'url': 'https://pokeapi.co/api/v2/ability/34/'}}], 'past_abilities': [{'generation': {'name': 'generation-iv', 'url': 'https://pokeapi.co/api/v2/generation/4/'}, 'abilities': [{'is_hidden': True, 'slot': 3, 'ability': None}]}], 'forms': [{'name': 'bulbasaur', 'url': 'https://pokeapi.co/api/v2/pokemon-form/1/'}], 'game_indices': [{'game_index': 153, 'version': {'name': 'red', 'url': 'https://pokeapi.co/api/v2/version/1/'}}, {'game_index': 153, 'version': {'name': 'blue', 'url': 'https://pokeapi.co/api/v2/version/2/'}}, {'game_index': 153, 'version': {'name': 'yellow', 'url': 'https://pokeapi.co/api

2026-10-01 12:11:54,129|[WARNING]|1929|136906103787648|dlt|validate.py|verify_normalized_table:113|In schema `quick_start`: The following columns in table 'detailed_info' did not receive any data during this load and therefore could not have their types inferred:
  - sprites__other__dream_world__front_female
  - sprites__other__official_artwork__versions__generation_ii__gold_and_silver__front_default
  - sprites__versions__generation_ix__scarlet_violet__front_female
  - sprites__versions__generation_vi__icons__front_female
  - sprites__versions__generation_vii__icons__front_female

Unless type hints are provided, these columns will not be materialized in the destination.
One way to provide type hints is to use the 'columns' argument in the '@dlt.resource' decorator.  For example:

@dlt.resource(columns={'sprites__other__dream_world__front_female': {'data_type': 'text'}})

2026-10-01 12:11:54,130|[WARNING]|1929|136906103787648|dlt|validate.py|verify_normalized_table:113|In schema `quick

Pipeline quick_start load step finished in 1.42 seconds
1 load package(s) were loaded to destination duckdb and into dataset pokedata_20261001121139
The duckdb destination used duckdb:////content/quick_start.duckdb location to store data
Load package 1790856713.7527692 is LOADED and contains no failed jobs


Alternatively, we could do:

In [27]:
@dlt.resource(table_name="pokemon")
def my_other_pokemons() -> TDataItems:
    pokemons = [
        {"id": "1", "name": "bulbasaur", "size": {"weight": 6.9, "height": 0.7}},
        {"id": "4", "name": "charmander", "size": {"weight": 8.5, "height": 0.6}},
        {"id": "25", "name": "pikachu", "size": {"weight": 6, "height": 0.4}},
    ]
    yield from pokemons


# NOTE: Transformer receives one item at a time
@dlt.transformer(data_from=my_other_pokemons, table_name="detailed_info")
def other_poke_details(
    data_item: TDataItem,
) -> TDataItems:
    item_id = data_item["id"]
    url = f"https://pokeapi.co/api/v2/pokemon/{item_id}"
    response = requests.get(url)
    details = response.json()

    yield details


load_info = another_pipeline.run(other_poke_details())
print(load_info)

2026-10-01 12:14:28,989|[WARNING]|1929|136906103787648|dlt|validate.py|verify_normalized_table:113|In schema `quick_start`: The following columns in table 'detailed_info' did not receive any data during this load and therefore could not have their types inferred:
  - sprites__other__dream_world__front_female
  - sprites__other__official_artwork__versions__generation_ii__gold_and_silver__front_default
  - sprites__versions__generation_ix__scarlet_violet__front_female
  - sprites__versions__generation_vi__icons__front_female
  - sprites__versions__generation_vii__icons__front_female

Unless type hints are provided, these columns will not be materialized in the destination.
One way to provide type hints is to use the 'columns' argument in the '@dlt.resource' decorator.  For example:

@dlt.resource(columns={'sprites__other__dream_world__front_female': {'data_type': 'text'}})

2026-10-01 12:14:28,990|[WARNING]|1929|136906103787648|dlt|validate.py|verify_normalized_table:113|In schema `quick

Pipeline quick_start load step finished in 0.87 seconds
1 load package(s) were loaded to destination duckdb and into dataset pokedata_20261001121139
The duckdb destination used duckdb:////content/quick_start.duckdb location to store data
Load package 1790856868.6651642 is LOADED and contains no failed jobs


You can also use pipe instead of `data_from`, this is useful when you want to apply `dlt.transformer` to multiple `dlt.resources`:

In [28]:
load_info = another_pipeline.run(my_pokemons | poke_details)
print(load_info)

Item: {'id': '1', 'name': 'bulbasaur', 'size': {'weight': 6.9, 'height': 0.7}}

Details: {'id': 1, 'name': 'bulbasaur', 'base_experience': 64, 'height': 7, 'is_default': True, 'order': 1, 'weight': 69, 'abilities': [{'is_hidden': False, 'slot': 1, 'ability': {'name': 'overgrow', 'url': 'https://pokeapi.co/api/v2/ability/65/'}}, {'is_hidden': True, 'slot': 3, 'ability': {'name': 'chlorophyll', 'url': 'https://pokeapi.co/api/v2/ability/34/'}}], 'past_abilities': [{'generation': {'name': 'generation-iv', 'url': 'https://pokeapi.co/api/v2/generation/4/'}, 'abilities': [{'is_hidden': True, 'slot': 3, 'ability': None}]}], 'forms': [{'name': 'bulbasaur', 'url': 'https://pokeapi.co/api/v2/pokemon-form/1/'}], 'game_indices': [{'game_index': 153, 'version': {'name': 'red', 'url': 'https://pokeapi.co/api/v2/version/1/'}}, {'game_index': 153, 'version': {'name': 'blue', 'url': 'https://pokeapi.co/api/v2/version/2/'}}, {'game_index': 153, 'version': {'name': 'yellow', 'url': 'https://pokeapi.co/api

2026-10-01 12:14:41,541|[WARNING]|1929|136906103787648|dlt|validate.py|verify_normalized_table:113|In schema `quick_start`: The following columns in table 'detailed_info' did not receive any data during this load and therefore could not have their types inferred:
  - sprites__other__dream_world__front_female
  - sprites__other__official_artwork__versions__generation_ii__gold_and_silver__front_default
  - sprites__versions__generation_ix__scarlet_violet__front_female
  - sprites__versions__generation_vi__icons__front_female
  - sprites__versions__generation_vii__icons__front_female

Unless type hints are provided, these columns will not be materialized in the destination.
One way to provide type hints is to use the 'columns' argument in the '@dlt.resource' decorator.  For example:

@dlt.resource(columns={'sprites__other__dream_world__front_female': {'data_type': 'text'}})

2026-10-01 12:14:41,542|[WARNING]|1929|136906103787648|dlt|validate.py|verify_normalized_table:113|In schema `quick

Pipeline quick_start load step finished in 0.85 seconds
1 load package(s) were loaded to destination duckdb and into dataset pokedata_20261001121139
The duckdb destination used duckdb:////content/quick_start.duckdb location to store data
Load package 1790856881.194373 is LOADED and contains no failed jobs


Check the loaded data:

In [29]:
# Query the 'detailed_info' table and convert the result to a DataFrame
another_pipeline.dataset().detailed_info.df()

,id,name,base_experience,height,is_default,order,weight,location_area_encounters,cries__latest,cries__legacy,...,sprites__versions__generation_v__black_white__animated__back_female,sprites__versions__generation_v__black_white__animated__front_female,sprites__versions__generation_v__black_white__animated__back_shiny_female,sprites__versions__generation_v__black_white__animated__front_shiny_female,sprites__other__showdown__back_female,sprites__other__showdown__front_female,sprites__other__showdown__back_shiny_female,sprites__other__showdown__front_shiny_female,sprites__other__home__front_female,sprites__other__home__front_shiny_female
0,1,bulbasaur,64,7,True,1,69,https://pokeapi.co/api/v2/pokemon/1/encounters,https://raw.githubusercontent.com/PokeAPI/crie...,https://raw.githubusercontent.com/PokeAPI/crie...,...,None,None,None,None,None,None,None,None,None,None
1,4,charmander,62,6,True,5,85,https://pokeapi.co/api/v2/pokemon/4/encounters,https://raw.githubusercontent.com/PokeAPI/crie...,https://raw.githubusercontent.com/PokeAPI/crie...,...,None,None,None,None,None,None,None,None,None,None
2,25,pikachu,112,4,True,35,60,https://pokeapi.co/api/v2/pokemon/25/encounters,https://raw.githubusercontent.com/PokeAPI/crie...,https://raw.githubusercontent.com/PokeAPI/crie...,...,https://raw.githubusercontent.com/PokeAPI/spri...,https://raw.githubusercontent.com/PokeAPI/spri...,https://raw.githubusercontent.com/PokeAPI/spri...,https://raw.githubusercontent.com/PokeAPI/spri...,https://raw.githubusercontent.com/PokeAPI/spri...,https://raw.githubusercontent.com/PokeAPI/spri...,https://raw.githubusercontent.com/PokeAPI/spri...,https://raw.githubusercontent.com/PokeAPI/spri...,https://raw.githubusercontent.com/PokeAPI/spri...,https://raw.githubusercontent.com/PokeAPI/spri...
3,1,bulbasaur,64,7,True,1,69,https://pokeapi.co/api/v2/pokemon/1/encounters,https://raw.githubusercontent.com/PokeAPI/crie...,https://raw.githubusercontent.com/PokeAPI/crie...,...,None,None,None,None,None,None,None,None,None,None
4,4,charmander,62,6,True,5,85,https://pokeapi.co/api/v2/pokemon/4/encounters,https://raw.githubusercontent.com/PokeAPI/crie...,https://raw.githubusercontent.com/PokeAPI/crie...,...,None,None,None,None,None,None,None,None,None,None
5,25,pikachu,112,4,True,35,60,https://pokeapi.co/api/v2/pokemon/25/encounters,https://raw.githubusercontent.com/PokeAPI/crie...,https://raw.githubusercontent.com/PokeAPI/crie...,...,https://raw.githubusercontent.com/PokeAPI/spri...,https://raw.githubusercontent.com/PokeAPI/spri...,https://raw.githubusercontent.com/PokeAPI/spri...,https://raw.githubusercontent.com/PokeAPI/spri...,https://raw.githubusercontent.com/PokeAPI/spri...,https://raw.githubusercontent.com/PokeAPI/spri...,https://raw.githubusercontent.com/PokeAPI/spri...,https://raw.githubusercontent.com/PokeAPI/spri...,https://raw.githubusercontent.com/PokeAPI/spri...,https://raw.githubusercontent.com/PokeAPI/spri...
6,1,bulbasaur,64,7,True,1,69,https://pokeapi.co/api/v2/pokemon/1/encounters,https://raw.githubusercontent.com/PokeAPI/crie...,https://raw.githubusercontent.com/PokeAPI/crie...,...,None,None,None,None,None,None,None,None,None,None
7,4,charmander,62,6,True,5,85,https://pokeapi.co/api/v2/pokemon/4/encounters,https://raw.githubusercontent.com/PokeAPI/crie...,https://raw.githubusercontent.com/PokeAPI/crie...,...,None,None,None,None,None,None,None,None,None,None
8,25,pikachu,112,4,True,35,60,https://pokeapi.co/api/v2/pokemon/25/encounters,https://raw.githubusercontent.com/PokeAPI/crie...,https://raw.githubusercontent.com/PokeAPI/crie...,...,https://raw.githubusercontent.com/PokeAPI/spri...,https://raw.githubusercontent.com/PokeAPI/spri...,https://raw.githubusercontent.com/PokeAPI/spri...,https://raw.githubusercontent.com/PokeAPI/spri...,https://raw.githubusercontent.com/PokeAPI/spri...,https://raw.githubusercontent.com/PokeAPI/spri...,https://raw.githubusercontent.com/PokeAPI/spri...,https://raw.githubusercontent.com/PokeAPI/spri...,https://raw.githubusercontent.

---
## **Reduce the nesting level of generated tables**
You can limit how deep dlt goes when generating nested tables and flattening dicts into columns. By default, the library will descend and generate nested tables for all nested lists, without limit.

You can set nesting level for all resources on the source level:

```python
@dlt.source(max_table_nesting=1)
def all_data():
  return my_df, get_genome_data, get_pokemon
```

or for each resource separately:

```python
@dlt.resource(table_name='pokemon_new', max_table_nesting=1)
def my_dict_list():
    yield data
```

In the example above, we want only 1 level of nested tables to be generated (so there are no nested tables of a nested table). Typical settings:

* `max_table_nesting=0` will not generate nested tables and will not flatten dicts into columns at all. All nested data will be represented as JSON.
* `max_table_nesting=1` will generate nested tables of root tables and nothing more. All nested data in nested tables will be represented as JSON.

---
## **Exercise 1: Create a pipeline for GitHub API – repos endpoint**

In this exercise, you'll build a dlt pipeline to fetch data from the GitHub REST API. The goal is to learn how to use `dlt.pipeline`, `dlt.resource`, and `dlt.source` to extract and load data into a destination.

## Instructions

1. **Explore the GitHub API**

   Visit the [GitHub REST API Docs](https://docs.github.com/en/rest) to understand the endpoint to [list public repositories](https://docs.github.com/en/rest/repos/repos?apiVersion=2022-11-28) for an organization:

   `GET https://api.github.com/orgs/{org}/repos`

2. **Build the pipeline**

   Write a script to:

   - Fetch repositories for the **dlt-hub** organization.
   - Use `dlt.resource` to define the data extraction logic.
   - Combine all resources into a single `@dlt.source`.
   - Load the data into a DuckDB database.

3. **Inspect the data**

   Use a `duckdb` connection, `sql_client`, or `pipeline.dataset()`.

> **Note**: For this exercise you don't need to use authentication or pagination.


Play with the API using the requests library:


In [30]:
from dlt.sources.helpers import requests

response = requests.get("https://api.github.com/orgs/dlt-hub/repos")
response.json()[0]

{'id': 427664318,
 'node_id': 'R_kgDOGX2jvg',
 'name': 'rasa_semantic_schema',
 'full_name': 'dlt-hub/rasa_semantic_schema',
 'private': False,
 'owner': {'login': 'dlt-hub',
  'id': 89419010,
  'node_id': 'MDEyOk9yZ2FuaXphdGlvbjg5NDE5MDEw',
  'avatar_url': 'https://avatars.githubusercontent.com/u/89419010?v=4',
  'gravatar_id': '',
  'url': 'https://api.github.com/users/dlt-hub',
  'html_url': 'https://github.com/dlt-hub',
  'followers_url': 'https://api.github.com/users/dlt-hub/followers',
  'following_url': 'https://api.github.com/users/dlt-hub/following{/other_user}',
  'gists_url': 'https://api.github.com/users/dlt-hub/gists{/gist_id}',
  'starred_url': 'https://api.github.com/users/dlt-hub/starred{/owner}{/repo}',
  'subscriptions_url': 'https://api.github.com/users/dlt-hub/subscriptions',
  'organizations_url': 'https://api.github.com/users/dlt-hub/orgs',
  'repos_url': 'https://api.github.com/users/dlt-hub/repos',
  'events_url': 'https://api.github.com/users/dlt-hub/events{/pr

In the code snippet below, you will find an **example** for the **`events`** endpoint:

In [39]:
import dlt
import requests


# Provided example: fetch GitHub events
@dlt.resource
def github_events():
    response = requests.get(
        "https://api.github.com/orgs/dlt-hub/events"
    )
    response.raise_for_status()
    yield from response.json()


# Your exercise: fetch GitHub repositories
@dlt.resource
def github_repos():
    response = requests.get(
        "https://api.github.com/orgs/dlt-hub/repos"
    )
    response.raise_for_status()
    yield from response.json()


# Group both resources
@dlt.source
def github_data():
    return github_events, github_repos


# Configure the pipeline
github_pipeline = dlt.pipeline(
    pipeline_name="github_pipeline",
    destination="duckdb",
    dataset_name="github_data"
)


# Load the data
load_info = github_pipeline.run(github_data())
print(load_info)

2026-10-01 12:36:24,940|[WARNING]|1929|136906103787648|dlt|validate.py|verify_normalized_table:113|In schema `github_data`: The following columns in table 'github_events' did not receive any data during this load and therefore could not have their types inferred:
  - payload__comment__minimized
  - payload__comment__performed_via_github_app
  - payload__comment__pin
  - payload__forkee__language
  - payload__forkee__license
  - payload__forkee__mirror_url
  - payload__issue__active_lock_reason
  - payload__issue__assignee
  - payload__issue__closed_at
  - payload__issue__milestone
  - payload__issue__performed_via_github_app
  - payload__issue__pinned_comment
  - payload__issue__pull_request__merged_at
  - payload__issue__state_reason
  - payload__issue__type
  - payload__review__body

Unless type hints are provided, these columns will not be materialized in the destination.
One way to provide type hints is to use the 'columns' argument in the '@dlt.resource' decorator.  For example:



Pipeline github_pipeline load step finished in 0.70 seconds
1 load package(s) were loaded to destination duckdb and into dataset github_data
The duckdb destination used duckdb:////content/github_pipeline.duckdb location to store data
Load package 1790858183.674332 is LOADED and contains no failed jobs


### Question
How many columns has the `github_repos` table? Use a `duckdb` connection, `sql_client` or `pipeline.dataset()`.

## **Exercise 2: Create a pipeline for the GitHub API – issue comments endpoint**

Create a `dlt.transformer` for the **"issue comments"** endpoint  
`https://api.github.com/repos/OWNER/REPO/issues/comments` for the `dlt-hub` organization.

Use the `github_repos` resource as the main resource for the transformer:

1. Get all repositories in the `dlt-hub` organization.  
2. Feed these repository names into the `dlt` transformer and retrieve the first page of issue comments for each `dlt-hub` repository.

> **Note**: As in Exercise 1, you don't need authentication or pagination. A single request per repository returns the first page of comments.


In [32]:
import dlt
import requests


@dlt.transformer(
    data_from=github_repos,
    table_name="github_issue_comments"
)
def github_issue_comments(repo):
    # Read the repository name from its dictionary
    repo_name = repo["name"]

    # Build the address for this repository's issue comments
    url = f"https://api.github.com/repos/dlt-hub/{repo_name}/issues/comments"

    # Request the first page of comments
    response = requests.get(url)
    response.raise_for_status()

    # Supply each comment separately to dlt
    yield from response.json()


# Fetch and load the comments
load_info = pipeline.run(github_issue_comments())
print(load_info)

# View the loaded table
pipeline.dataset().github_issue_comments.df()

NameError: name 'github_repos' is not defined

### Question
How many columns has the `github_issue_comment` table? Use a `duckdb` connection, `sql_client` or `pipeline.dataset()`.

✅ ▶ Proceed to the [next lesson](https://github.com/dlt-hub/dlt/blob/master/docs/education/dlt-fundamentals-course/lesson_3_pagination_and_authentication_and_dlt_configuration.ipynb)!